<a href="https://colab.research.google.com/github/mohid-arif/Semeval-Task-1-MWAHAHA/blob/main/Assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
!pip install -q --upgrade transformers datasets accelerate

In [ ]:
import transformers
print(transformers.__version__)

4.57.3


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="output.jsonl",
    split="train"
)

print(dataset)
print(dataset[0])


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 20208
})
{'prompt': 'Words: far, magnet\nJoke:', 'completion': " I'm very pleased with my new fridge magnet. So far I've got twelve fridges."}


In [ ]:
dataset = dataset.train_test_split(test_size=0.05, seed=42)

train_ds = dataset["train"]
val_ds = dataset["test"]

print(len(train_ds), len(val_ds))


19197 1011


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

MODEL_NAME = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # important

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))


Embedding(50257, 768)

In [ ]:
MAX_LENGTH = 256

def tokenize(example):
    text = example["prompt"] + example["completion"]
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_ds = train_ds.map(tokenize, remove_columns=train_ds.column_names)
val_ds = val_ds.map(tokenize, remove_columns=val_ds.column_names)


In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-jokes",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2
)



In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

trainer.train()


/tmp/ipython-input-1713228816.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,0.816700
100,0.464300
150,0.458500
200,0.448700
250,0.446500
300,0.451500
350,0.450300
400,0.444200
450,0.447200
500,0.423300


Step,Training Loss
50,0.816700
100,0.464300
150,0.458500
200,0.448700
250,0.446500
300,0.451500
350,0.450300
400,0.444200
450,0.447200
500,0.423300


TrainOutput(global_step=3600, training_loss=0.39027004718780517, metrics={'train_runtime': 5039.6868, 'train_samples_per_second': 11.427, 'train_steps_per_second': 0.714, 'total_flos': 7524034707456000.0, 'train_loss': 0.39027004718780517, 'epoch': 3.0})

In [ ]:
trainer.save_model("gpt2-jokes-final")
tokenizer.save_pretrained("gpt2-jokes-final")


('gpt2-jokes-final/tokenizer_config.json',
 'gpt2-jokes-final/special_tokens_map.json',
 'gpt2-jokes-final/vocab.json',
 'gpt2-jokes-final/merges.txt',
 'gpt2-jokes-final/added_tokens.json')

In [ ]:
import shutil

shutil.make_archive("gpt2-jokes-final", "zip", "gpt2-jokes-final")
files.download("gpt2-jokes-final.zip")


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2-jokes-final",
    tokenizer=tokenizer,
    device=0
)

prompt = "Headline: Government promises reform\nJoke:"

out = generator(
    prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.9,
    top_p=0.95
)

print(out[0]["generated_text"])


Device set to use cuda:0


Headline: Government promises reform
Joke: We won't let you down! /s


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving task-a-en.tsv to task-a-en.tsv


In [ ]:
import pandas as pd
from transformers import GPT2Tokenizer, GPT2LMHeadModel, pipeline
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
MODEL_DIR = "gpt2-jokes-final"    # path to your fine-tuned model
TEST_FILE = "test.tsv"            # your test dataset
OUTPUT_FILE = "test_predictions.tsv"
MAX_NEW_TOKENS = 50
TEMPERATURE = 0.9
TOP_P = 0.95

# ==============================
# LOAD MODEL & TOKENIZER
# ==============================
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0  # change to -1 for CPU
)

# ==============================
# LOAD TEST DATA
# ==============================
# Assuming TSV with columns: word1, word2, headline
test_df = pd.read_csv(TEST_FILE, sep="\t")

# Determine prompts
def make_prompt(row):
    word1, word2, headline = row.get("word1", ""), row.get("word2", ""), row.get("headline", "")
    if pd.notna(headline) and headline != "-":
        return f"Headline: {headline}\nJoke:"
    elif pd.notna(word1) and pd.notna(word2) and word1 != "-" and word2 != "-":
        return f"Words: {word1}, {word2}\nJoke:"
    else:
        return None

test_df["prompt"] = test_df.apply(make_prompt, axis=1)
test_df = test_df[test_df["prompt"].notna()]

# ==============================
# GENERATE JOKES
# ==============================
generated_jokes = []

for prompt in tqdm(test_df["prompt"], desc="Generating jokes"):
    output = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.eos_token_id
    )
    # Extract generated text after the prompt
    text = output[0]["generated_text"]
    joke = text[len(prompt):].strip()
    generated_jokes.append(joke)

# ==============================
# SAVE TO FILE
# ==============================
test_df["generated_joke"] = generated_jokes
test_df.to_csv(OUTPUT_FILE, sep="\t", index=False)

print(f"Saved generated jokes to {OUTPUT_FILE}")


Device set to use cuda:0
Generating jokes: 100%|██████████| 1200/1200 [03:59<00:00,  5.01it/s]

Saved generated jokes to test_predictions.tsv
